In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Literal, Optional

import numpy as np
from numpy.typing import NDArray
from PIL import Image

# Make an alias for the image format type to save space
ImageF = NDArray[np.float32]  # HxWxC in [0,1]

In [2]:
def load_image_rgb(path: str | Path) -> ImageF:
    """Load an image in RGB from a specified file path."""
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

def save_image_rgb(path: str | Path, img: ImageF) -> None:
    """Save an image in RGB to a specified file path."""
    img8 = np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)
    Image.fromarray(img8, mode="RGB").save(path)

In [ ]:
def rgb_to_luma(img: ImageF) -> ImageF:
    """Conform an RGB image to Rec.709's luma coefficients."""
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    y = 0.2126*r + 0.7152*g + 0.0722*b
    return y.astype(np.float32)

def luma_to_rgb(y: ImageF) -> ImageF:
    return np.stack([y, y, y], axis=-1).astype(np.float32)

def srgb_to_linear(x: ImageF) -> ImageF:
    """Piecewise implementation of the sRGB inverse EOTF."""
    a = 0.055
    return np.where(
        x <= 0.04045,
        x / 12.92,
        ((x + a) / (1.0 + a)) ** 2.4,
    ).astype(np.float32)